# CrewAI + OpenRouter (gpt-oss-20b:free) + AgentOps Monitoring

This notebook builds a simple **2-agent crew** (Researcher + Writer) using **CrewAI**,
powered by a **free model on OpenRouter**, and fully monitored with **AgentOps**.

### What you will learn
- How to route LLM calls through OpenRouter using a free model
- How to build a multi-agent workflow with CrewAI
- How to instrument everything with AgentOps for observability (session replay, cost tracking, failure detection)

### Prerequisites (get these before running)
1. **OpenRouter API Key** — free account at [openrouter.ai/keys](https://openrouter.ai/keys). The model `openai/gpt-oss-20b:free` is free to use (rate-limited).
2. **AgentOps API Key** — free account at [app.agentops.ai](https://app.agentops.ai), found under Settings.

> Keep both keys secret. In Colab, it is best practice to use the built-in **Secrets** manager (key icon on the left sidebar) instead of pasting keys directly into cells.

> **Important:** if you see `Upload failed: 401` in the output later, it means the `AGENTOPS_API_KEY` cell still has the placeholder value (or the wrong key) — double-check you replaced it with your real key from app.agentops.ai before running.


## 1. Install dependencies

In [ ]:
!pip install -q crewai agentops

## 2. Set your API keys

Option A — paste directly (quick, but not recommended for shared notebooks):


In [ ]:
import os

os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-XXXXXXXXXXXXXXXXXXXXXXXX"  # <-- your OpenRouter key
os.environ["AGENTOPS_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXX"            # <-- your AgentOps key

Option B — using Colab Secrets (recommended):
```python
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["AGENTOPS_API_KEY"] = userdata.get("AGENTOPS_API_KEY")
```


## 3. Initialize AgentOps

This **must** run before creating any Agent/LLM/Crew objects. AgentOps auto-instruments
CrewAI and the underlying LLM client, so every call gets logged automatically — no manual logging needed.


In [ ]:
import agentops

agentops.init(
    tags=["openrouter-demo", "gpt-oss-20b-free"],
    auto_start_session=True,
)

## 4. Configure the model via OpenRouter

CrewAI uses LiteLLM under the hood, so the model name needs the `openrouter/` prefix,
followed by the exact model slug as listed on openrouter.ai/models.


In [ ]:
from crewai import Agent, Task, Crew, LLM

llm = LLM(
    model="openrouter/openai/gpt-oss-20b:free",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

## 5. Define the agents

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="Gather accurate, focused information on the requested topic",
    backstory="An expert technical researcher who reasons clearly and produces well-structured findings.",
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Writer",
    goal="Turn research findings into a clear, simple article for a general audience",
    backstory="A professional technical writer who writes in an accessible, direct style.",
    llm=llm,
    verbose=True,
)

## 6. Define the tasks

In [ ]:
research_task = Task(
    description="Research and summarize the top 5 current trends in AI Agents during 2026.",
    expected_output="A list of 5 points, each with a one- or two-sentence explanation.",
    agent=researcher,
)

writing_task = Task(
    description="Using the research findings, write a short article (around 200-300 words) explaining these trends in simple terms.",
    expected_output="A coherent 200-300 word article.",
    agent=writer,
    context=[research_task],
)

## 7. Assemble and run the crew

In [ ]:
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=True,
)

# NOTE: Colab (and Jupyter in general) already runs inside an active asyncio
# event loop. Calling the synchronous crew.kickoff() from inside that loop
# raises: "Agent execution was invoked synchronously from within a running
# event loop". The fix is to use kickoff_async() with await instead
# (top-level await works directly in a notebook cell).
try:
    result = await crew.kickoff_async()
    print(result)
    try:
        agentops.end_trace(end_state="Success")  # newer AgentOps API (end_session is deprecated)
    except Exception:
        pass
except Exception as e:
    try:
        agentops.end_trace(end_state="Failed")
    except Exception:
        pass
    raise

## 8. Review the run in AgentOps

Open [app.agentops.ai](https://app.agentops.ai) and open the latest session. You should see:

- **Session timeline** — the full sequence of events (which agent ran when)
- **Every LLM call** — the exact prompt sent, the response received, token counts, and cost per call (will show $0 since the model is free, but the tracking structure is identical to a paid model)
- **Agent-to-agent flow** — how the `context=[research_task]` handoff moved data from the Researcher to the Writer
- **Errors/failures** — clearly flagged if any call times out or fails
- **Replay** — you can rewind to any point in the session and inspect the exact state

### Notes on the free model
- `openai/gpt-oss-20b:free` is rate-limited (requests per minute/day). If you hit a `429` error, wait a minute and retry.
- The model name must match exactly: `openrouter/openai/gpt-oss-20b:free`.
